In [1]:
!pip install numpy pandas scipy mne torch torchvision torchaudio scikit-learn matplotlib seaborn tqdm


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

2.9.0.dev20250821+rocm7.0.0.git125803b7
True



In [3]:
import os
import time
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

import numpy as np
import pandas as pd
from scipy.signal import resample
import mne
mne.set_log_level('WARNING')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── ROCm MI300X OPTIMIZATIONS ──
os.environ['HSA_ENABLE_SDMA'] = '0'
os.environ['ROCR_VISIBLE_DEVICES'] = '0'
os.environ['HIP_LAUNCH_BLOCKING'] = '0'
os.environ['PYTORCH_HIP_ALLOC_CONF'] = 'max_split_size_mb:512'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch Version : {torch.__version__}")
print(f"Active Device   : {DEVICE}")

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


PyTorch Version : 2.9.0.dev20250821+rocm7.0.0.git125803b7
Active Device   : cuda


In [4]:
import os
import subprocess

dataset_dir = "ds004504"

if not os.path.exists(dataset_dir) or not os.path.exists(f"{dataset_dir}/participants.tsv"):
    print(f"Downloading ds004504 dataset from OpenNeuro...")
    
    # Ensure AWS CLI is installed (required for OpenNeuro downloads)
    try:
        subprocess.run(["aws", "--version"], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except FileNotFoundError:
        print("AWS CLI not found. Installing via pip...")
        subprocess.run(["pip", "install", "awscli"], check=True)
        
    os.makedirs(dataset_dir, exist_ok=True)
    
    # Sync the dataset from the public S3 bucket
    subprocess.run([
        "aws", "s3", "sync", "--no-sign-request", 
        "s3://openneuro.org/ds004504", 
        f"./{dataset_dir}"
    ])
    print("Download complete!")
else:
    print(f"Dataset already exists at ./{dataset_dir}. Skipping download.")


AWS CLI not found. Installing via pip...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 40.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 161.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 92.0 MB/s  0:00:00
  Attempting uninstall: botocorem━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/6 [docutils]
    Found existing installation: botocore 1.35.99━━━━━━━━━━━━━ 1/6 [docutils]
    Uninstalling botocore-1.35.99:━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/6 [docutils]
      Successfully uninstalled botocore-1.35.99━━━━━━━━━━━━━━━━━━━ 3/6 [botocore]
  Attempting uninstall: s3transfer90m╺━━━━━━━━━━━━━━━━━━━ 3/6 [botocore]
    Found existing installation: s3transfer 0.10.4━━━━━━━━━━━━ 3/6 [botocore]
    Uninstalling s3transfer-0.10.4:━━━━━━━━━━━━━━━━━━━ 3/6 [botocore]
      Successfully uninstalled s3transfer-0.10.4━━━━━━━━━━━━━━ 3/6 [botocore]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [awscli]2m5/6 [awscli]


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
boto3 1.35.42 requires botocore<1.36.0,>=1.35.42, but you have botocore 1.43.6 which is incompatible.
boto3 1.35.42 requires s3transfer<0.11.0,>=0.10.0, but you have s3transfer 0.17.0 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


download: s3://openneuro.org/ds004504/CHANGES to ds004504/CHANGES 
download: s3://openneuro.org/ds004504/.datalad/config to ds004504/.datalad/config
download: s3://openneuro.org/ds004504/README to ds004504/README   
download: s3://openneuro.org/ds004504/dataset_description.json to ds004504/dataset_description.json
download: s3://openneuro.org/ds004504/.gitattributes to ds004504/.gitattributes
download: s3://openneuro.org/ds004504/derivatives/sub-001/eeg/sub-001_task-eyesclosed_eeg.set to ds004504/derivatives/sub-001/eeg/sub-001_task-eyesclosed_eeg.set
download: s3://openneuro.org/ds004504/derivatives/sub-003/eeg/sub-003_task-eyesclosed_eeg.set to ds004504/derivatives/sub-003/eeg/sub-003_task-eyesclosed_eeg.set
download: s3://openneuro.org/ds004504/derivatives/sub-004/eeg/sub-004_task-eyesclosed_eeg.set to ds004504/derivatives/sub-004/eeg/sub-004_task-eyesclosed_eeg.set
download: s3://openneuro.org/ds004504/derivatives/sub-005/eeg/sub-005_task-eyesclosed_eeg.set to ds004504/derivatives/

In [5]:
BASE_DIR = Path(os.getcwd())

DS004_DATA_DIR = BASE_DIR / 'ds004504'
FSU_DATA_DIR   = BASE_DIR / 'FSU_data'

COMMON_CHANNELS = [
    'Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2',
    'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'Fz', 'Cz', 'Pz'
]

SFREQ = 250.0       # Target Frequency for both datasets
EPOCH_SEC = 4       # Strict 4-second epochs
EPOCH_SAMPLES = int(SFREQ * EPOCH_SEC) # 1000 samples

def chunk_into_epochs(data):
    """Strictly slice continuous data into non-overlapping 4-second epochs"""
    n_epochs = data.shape[1] // EPOCH_SAMPLES
    if n_epochs == 0:
        return []
    # Drop trailing data that doesn't fit into a full 4-second block
    data = data[:, :n_epochs * EPOCH_SAMPLES]
    # Reshape to (n_epochs, n_channels, 1000)
    epochs = np.stack(np.split(data, n_epochs, axis=1))
    return epochs

def load_ds004504(data_dir):
    df = pd.read_csv(data_dir / 'participants.tsv', sep='\t').dropna(subset=['Group'])
    df['label'] = df['Group'].map({'A': 0, 'C': 1})
    deriv_dir = data_dir / 'derivatives'
    
    X_all, y_all = [], []
    
    for _, row in tqdm(df.dropna(subset=['label']).iterrows(), desc="DS004504 (Train)"):
        sid, label = row['participant_id'], int(row['label'])
        fpath = deriv_dir / sid / 'eeg' / f'{sid}_task-eyesclosed_eeg.set'
        
        if fpath.exists():
            raw = mne.io.read_raw_eeglab(str(fpath), preload=True, verbose=False)
            raw.pick_channels(COMMON_CHANNELS).reorder_channels(COMMON_CHANNELS)
            
            # STRICT RESAMPLING
            if raw.info['sfreq'] != SFREQ:
                raw.resample(SFREQ)
                
            data = raw.get_data().astype(np.float32)
            epochs = chunk_into_epochs(data)
            
            if len(epochs) > 0:
                X_all.extend(epochs)
                y_all.extend([label] * len(epochs))
                
    return np.array(X_all), np.array(y_all)

def load_fsu(data_dir):
    # Group FSU text files by Patient
    patient_files = defaultdict(dict)
    for path in data_dir.rglob('*.txt'):
        parts = path.parts
        # Assuming structure: FSU_data/Class/Condition/Patient/Channel.txt
        # We only want Eyes Closed (often 'EC' or 'Eyes_closed')
        if 'open' in str(path).lower(): continue 
            
        cls_name = parts[-4]
        label = 0 if 'AD' in cls_name.upper() else 1
        patient = parts[-2]
        ch = path.stem
        
        if ch in COMMON_CHANNELS:
            patient_files[(label, patient)][ch] = path
            
    X_all, y_all = [], []
    for (label, patient), ch_dict in tqdm(patient_files.items(), desc="FSU (Test)"):
        # Ensure patient has all 19 channels
        if len(ch_dict) == len(COMMON_CHANNELS):
            ch_data = []
            for ch in COMMON_CHANNELS:
                arr = np.loadtxt(ch_dict[ch], dtype=np.float32)
                ch_data.append(arr)
                
            data = np.stack(ch_data) # Shape: (19, N_samples)
            
            # STRICT RESAMPLING: FSU is 128Hz, we must upscale to 250Hz mathematically
            orig_sfreq = 128.0
            target_samples = int(data.shape[1] * (SFREQ / orig_sfreq))
            data_resampled = resample(data, target_samples, axis=1).astype(np.float32)
            
            epochs = chunk_into_epochs(data_resampled)
            if len(epochs) > 0:
                X_all.extend(epochs)
                y_all.extend([label] * len(epochs))
                
    return np.array(X_all), np.array(y_all)

# Execute Data Extraction
X_tr, y_tr = load_ds004504(DS004_DATA_DIR)
X_te, y_te = load_fsu(FSU_DATA_DIR)

if len(X_te) == 0:
    raise ValueError("FSU Dataset loaded 0 epochs. Check your FSU_DATA_DIR path!")

print(f"Train Epochs (DS004504): {X_tr.shape}")
print(f"Test Epochs (FSU):       {X_te.shape}")

# ── STRICT NORMALIZATION (Preventing Data Leakage) ──
scaler = StandardScaler()
# Fit only on training data (flattened)
X_tr_flat = X_tr.reshape(X_tr.shape[0], -1)
X_te_flat = X_te.reshape(X_te.shape[0], -1)

X_tr_scaled = scaler.fit_transform(X_tr_flat)
X_te_scaled = scaler.transform(X_te_flat)

# Reshape back to (N, 1, 19, 1000) for CNN processing
X_tr = X_tr_scaled.reshape(-1, 1, len(COMMON_CHANNELS), EPOCH_SAMPLES)
X_te = X_te_scaled.reshape(-1, 1, len(COMMON_CHANNELS), EPOCH_SAMPLES)

class EEGDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.x[i], self.y[i]

train_loader = DataLoader(EEGDataset(X_tr, y_tr), batch_size=64, shuffle=True)
test_loader  = DataLoader(EEGDataset(X_te, y_te), batch_size=64, shuffle=False)


DS004504 (Train): 65it [00:20,  3.19it/s]
FSU (Test): 100% 92/92 [00:00<00:00, 399.93it/s]


Train Epochs (DS004504): (13282, 19, 1000)
Test Epochs (FSU):       (184, 19, 1000)


In [6]:
class GradientReversalLayer(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

def grad_reverse(x, alpha=1.0):
    return GradientReversalLayer.apply(x, alpha)

class SIREEGNet(nn.Module):
    """Subject-Invariant Representation EEGNet (SIR-EEGNet)"""
    def __init__(self, n_channels=19, n_samples=1000, n_classes=2, F1=8, D=2, F2=16):
        super().__init__()
        
        # ── Shared Feature Extractor ──
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, 125), padding=(0, 62), bias=False),
            nn.BatchNorm2d(F1),
            nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(0.5)
        )
        
        self.block2 = nn.Sequential(
            nn.Conv2d(F1 * D, F2, (1, 16), padding=(0, 8), groups=F1 * D, bias=False),
            nn.Conv2d(F2, F2, (1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(0.5)
        )
        
        # Calculate flattened dimension
        out_dim = F2 * (n_samples // 32)
        
        # ── Main Classifier (AD vs HC) ──
        self.classifier = nn.Linear(out_dim, n_classes)
        
        # ── Domain Discriminator (Subject Invariance) ──
        # Predicts which dataset/subject the signal came from. 
        # The GRL negates gradients to make the feature extractor blind to domains.
        self.domain_classifier = nn.Sequential(
            nn.Linear(out_dim, 64),
            nn.ELU(),
            nn.Linear(64, 2) # 2 Domains: Source (DS004) vs Target (FSU)
        )

    def forward(self, x, alpha=None):
        features = self.block2(self.block1(x))
        features = features.view(features.size(0), -1)
        
        # Primary Output
        class_output = self.classifier(features)
        
        # Domain Output (only used during Domain Adversarial Training)
        if alpha is not None:
            rev_features = grad_reverse(features, alpha)
            domain_output = self.domain_classifier(rev_features)
            return class_output, domain_output
            
        return class_output


In [ ]:
model = SIREEGNet(n_channels=len(COMMON_CHANNELS), n_samples=EPOCH_SAMPLES).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Handle class imbalances
classes, counts = np.unique(y_tr, return_counts=True)
weight = torch.tensor([len(y_tr) / (2 * c) for c in counts], dtype=torch.float32).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=weight)

scaler_amp = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None
EPOCHS = 60

print("=" * 60)
print(f"Training SIR-EEGNet on DS004504 (Standardized Epochs)...")
print("=" * 60)

model.train()
for epoch in range(EPOCHS):
    ep_loss = 0
    for x_b, y_b in train_loader:
        x_b, y_b = x_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        
        if scaler_amp:
            with torch.amp.autocast('cuda'):
                preds = model(x_b)
                loss = criterion(preds, y_b)
            scaler_amp.scale(loss).backward()
            scaler_amp.step(optimizer)
            scaler_amp.update()
        else:
            preds = model(x_b)
            loss = criterion(preds, y_b)
            loss.backward()
            optimizer.step()
            
        ep_loss += loss.item()
        
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1:03d}/{EPOCHS}] | Loss: {ep_loss/len(train_loader):.4f}")
print("\n" + "=" * 60)
print("Evaluating SIR-EEGNet on Fully Resampled FSU Dataset")
print("=" * 60)

model.eval()
true_labels, pred_labels = [], []

with torch.no_grad():
    for x_b, y_b in test_loader:
        x_b = x_b.to(DEVICE)
        
        if scaler_amp:
            with torch.amp.autocast('cuda'):
                preds = model(x_b)
        else:
            preds = model(x_b)
            
        pred_labels.extend(preds.argmax(dim=1).cpu().numpy().tolist())
        true_labels.extend(y_b.numpy().tolist())

acc = accuracy_score(true_labels, pred_labels)
f1_m = f1_score(true_labels, pred_labels, average='macro', zero_division=0)

print(f"Cross-Dataset Accuracy : {acc*100:.2f}%")
print(f"Cross-Dataset F1-Macro : {f1_m*100:.2f}%")
print("\nClassification Report:\n")
print(classification_report(true_labels, pred_labels, target_names=['AD', 'HC'], zero_division=0))

# ── Visualization ──
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(true_labels, pred_labels, labels=[0, 1])

sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=ax, xticklabels=['AD', 'HC'], yticklabels=['AD', 'HC'], cbar=False)
ax.set_title(f"SIR-EEGNet: Train DS004504 → Test FSU\nAccuracy: {acc*100:.1f}%", fontweight='bold')
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.show()


Training SIR-EEGNet on DS004504 (Standardized Epochs)...
Epoch [010/60] | Loss: 0.2062
Epoch [020/60] | Loss: 0.1545
